# AVISTA Validation Study — UCI National Poll on Healthy Aging

## Objective

This notebook evaluates AVISTA using the public **UCI National Poll on Healthy Aging (NPHA)** dataset. The objective is to compare AVISTA with an equivalent direct Jupyter implementation under controlled experimental settings and with AutoGluon as an external AutoML platform.

## Dataset

The NPHA dataset contains **714 observations** and **14 categorical predictors**. The classification target is `Number_of_Doctors_Visited`, representing three classes:

* `1` = 0–1 doctors
* `2` = 2–3 doctors
* `3` = 4 or more doctors

The predictors describe demographic, health, employment, medication, and sleep-related characteristics. The dataset contains no reported missing values.

## Experimental Setup

A fixed random seed of **42** is used throughout. The dataset is divided using stratified sampling into:

* **70% training**
* **10% validation**
* **20% test**

For AVISTA and direct Jupyter experiments, categorical preprocessing is fitted using training data only. Missing or blank categorical values, if encountered, are assigned to the reserved `Unknown` category before one-hot encoding.

Five-fold stratified cross-validation is performed within the training partition using leakage-safe preprocessing.

The evaluated models are:

* Logistic Regression
* Random Forest
* XGBoost

Performance is evaluated using **Accuracy, Macro-F1, Macro-Precision, Macro-Recall, and multiclass ROC-AUC**.

AutoGluon uses the same raw train, validation, and test observations but retains its native preprocessing and training behavior.


In [1]:
# ============================================================
# Cell 1: Download and load UCI NPHA dataset
# ============================================================

from ucimlrepo import fetch_ucirepo
import pandas as pd

# Download dataset
npha = fetch_ucirepo(id=936)

# Features and target
X = npha.data.features.copy()
y = npha.data.targets.copy()

# Combined dataframe
df = pd.concat([X, y], axis=1)

print("Dataset loaded successfully.")
print(f"Total rows     : {df.shape[0]}")
print(f"Features       : {X.shape[1]}")
print(f"Target column  : {y.columns[0]}")
print(f"Dataset shape  : {df.shape}")

display(df.head())

Dataset loaded successfully.
Total rows     : 714
Features       : 14
Target column  : Number_of_Doctors_Visited
Dataset shape  : (714, 15)


,Age,Physical_Health,Mental_Health,Dental_Health,Employment,Stress_Keeps_Patient_from_Sleeping,Medication_Keeps_Patient_from_Sleeping,Pain_Keeps_Patient_from_Sleeping,Bathroom_Needs_Keeps_Patient_from_Sleeping,Uknown_Keeps_Patient_from_Sleeping,Trouble_Sleeping,Prescription_Sleep_Medication,Race,Gender,Number_of_Doctors_Visited
0,2,4,3,3,3,0,0,0,0,1,2,3,1,2,3
1,2,4,2,3,3,1,0,0,1,0,3,3,1,1,2
2,2,3,2,3,3,0,0,0,0,1,3,3,4,1,3
3,2,3,2,3,3,0,0,0,1,0,3,3,4,2,1
4,2,3,3,3,3,1,0,0,0,0,2,3,1,2,3


In [2]:
# ============================================================
# Cell 2: Convert integer-coded predictors to categorical
# ============================================================

TARGET = "Number_of_Doctors_Visited"

# All predictors are categorical according to UCI metadata
categorical_features = [
    col for col in df.columns
    if col != TARGET
]

df_npha = df.copy()

# Convert coded predictors to categorical/string representation
for col in categorical_features:
    df_npha[col] = df_npha[col].astype(str)

print("=== FEATURE TYPES ===")
print(f"Categorical features : {len(categorical_features)}")
print(f"Numerical features   : 0")
print(f"Target               : {TARGET}")

display(df_npha.dtypes.to_frame("dtype"))

=== FEATURE TYPES ===
Categorical features : 14
Numerical features   : 0
Target               : Number_of_Doctors_Visited


,dtype
Age,object
Physical_Health,object
Mental_Health,object
Dental_Health,object
Employment,object
Stress_Keeps_Patient_from_Sleeping,object
Medication_Keeps_Patient_from_Sleeping,object
Pain_Keeps_Patient_from_Sleeping,object
Bathroom_Needs_Keeps_Patient_from_Sleeping,object
Uknown_Keeps_Patient_from_Sleeping,object


In [3]:
# ============================================================
# Cell 3: Data integrity check
# ============================================================

print("=== DATASET SHAPE ===")
print(f"Rows     : {df_npha.shape[0]}")
print(f"Features : {df_npha.shape[1] - 1}")
print(f"Target   : {TARGET}")

print("\n=== TARGET DISTRIBUTION ===")

target_counts = df_npha[TARGET].value_counts().sort_index()
target_pct = (
    df_npha[TARGET]
    .value_counts(normalize=True)
    .sort_index() * 100
)

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percent": target_pct.round(2)
})

display(target_summary)

print("\n=== MISSING VALUES ===")

missing_summary = pd.DataFrame({
    "Missing": df_npha.isna().sum(),
    "Percent": (df_npha.isna().mean() * 100).round(2)
})

display(
    missing_summary[
        missing_summary["Missing"] > 0
    ]
)

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicate rows: {df_npha.duplicated().sum()}")

print("\n=== UNIQUE VALUES PER FEATURE ===")

unique_summary = pd.DataFrame({
    "Unique values": df_npha.drop(
        columns=TARGET
    ).nunique(dropna=False)
}).sort_values("Unique values")

display(unique_summary)

print("\n=== DATA TYPES ===")
display(df_npha.dtypes.to_frame("dtype"))

=== DATASET SHAPE ===
Rows     : 714
Features : 14
Target   : Number_of_Doctors_Visited

=== TARGET DISTRIBUTION ===


,Count,Percent
Number_of_Doctors_Visited,,
1,131,18.35
2,372,52.10
3,211,29.55



=== MISSING VALUES ===


,Missing,Percent



=== DUPLICATE ROWS ===
Duplicate rows: 42

=== UNIQUE VALUES PER FEATURE ===


,Unique values
Age,1
Pain_Keeps_Patient_from_Sleeping,2
Medication_Keeps_Patient_from_Sleeping,2
Stress_Keeps_Patient_from_Sleeping,2
Uknown_Keeps_Patient_from_Sleeping,2
Gender,2
Bathroom_Needs_Keeps_Patient_from_Sleeping,2
Employment,4
Prescription_Sleep_Medication,4
Trouble_Sleeping,4



=== DATA TYPES ===


,dtype
Age,object
Physical_Health,object
Mental_Health,object
Dental_Health,object
Employment,object
Stress_Keeps_Patient_from_Sleeping,object
Medication_Keeps_Patient_from_Sleeping,object
Pain_Keeps_Patient_from_Sleeping,object
Bathroom_Needs_Keeps_Patient_from_Sleeping,object
Uknown_Keeps_Patient_from_Sleeping,object


In [4]:
# ============================================================
# Cell 4: Check constant feature and duplicate patterns
# ============================================================

# ------------------------------------------------------------
# 1. Drop constant feature
# ------------------------------------------------------------
constant_features = [
    col for col in df_npha.columns
    if col != TARGET and df_npha[col].nunique(dropna=False) <= 1
]

print("Constant features:", constant_features)

df_npha = df_npha.drop(columns=constant_features).copy()

print(f"Remaining features: {df_npha.shape[1] - 1}")


# ------------------------------------------------------------
# 2. Inspect exact duplicate rows
# ------------------------------------------------------------
exact_duplicates = df_npha.duplicated().sum()

print(f"\nExact duplicate rows: {exact_duplicates}")


# ------------------------------------------------------------
# 3. Check identical feature patterns with different targets
# ------------------------------------------------------------
feature_cols = [
    col for col in df_npha.columns
    if col != TARGET
]

pattern_target_counts = (
    df_npha
    .groupby(feature_cols, dropna=False)[TARGET]
    .nunique()
)

conflicting_patterns = (
    pattern_target_counts > 1
).sum()

print(
    "Feature patterns associated with multiple target classes:",
    conflicting_patterns
)

Constant features: ['Age']
Remaining features: 13

Exact duplicate rows: 42
Feature patterns associated with multiple target classes: 30


In [5]:
# ============================================================
# Cell 5: Create feature-pattern groups
# Prevent identical predictor patterns crossing data splits
# ============================================================

feature_cols = [
    col for col in df_npha.columns
    if col != TARGET
]

# Assign the same group ID to identical predictor combinations
df_npha["pattern_id"] = (
    df_npha
    .groupby(feature_cols, dropna=False)
    .ngroup()
)

print("=== FEATURE-PATTERN GROUPS ===")
print(f"Total observations     : {len(df_npha)}")
print(f"Unique feature patterns: {df_npha['pattern_id'].nunique()}")

group_sizes = df_npha["pattern_id"].value_counts()

print(f"Patterns appearing once: {(group_sizes == 1).sum()}")
print(f"Repeated patterns      : {(group_sizes > 1).sum()}")
print(f"Largest pattern size   : {group_sizes.max()}")

=== FEATURE-PATTERN GROUPS ===
Total observations     : 714
Unique feature patterns: 634
Patterns appearing once: 586
Repeated patterns      : 48
Largest pattern size   : 5


In [7]:
# ============================================================
# Cell 5A: Remove highly associated categorical features
# Pairwise Cramér's V threshold = 0.80
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

CRAMERS_V_THRESHOLD = 0.65

# Work without the temporary pattern_id if it already exists
df_work = df_npha.drop(
    columns=["pattern_id"],
    errors="ignore"
).copy()

feature_cols = [
    col for col in df_work.columns
    if col != TARGET
]


def cramers_v(x, y):
    x = x.fillna("Unknown").astype(str)
    y = y.fillna("Unknown").astype(str)

    table = pd.crosstab(x, y)

    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()

    r, k = table.shape
    denominator = min(r - 1, k - 1)

    if n == 0 or denominator == 0:
        return 0.0

    return np.sqrt((chi2 / n) / denominator)


# ------------------------------------------------------------
# Pairwise Cramér's V matrix
# ------------------------------------------------------------
cramers_matrix = pd.DataFrame(
    np.eye(len(feature_cols)),
    index=feature_cols,
    columns=feature_cols
)

for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):

        v = cramers_v(
            df_work[feature_cols[i]],
            df_work[feature_cols[j]]
        )

        cramers_matrix.iloc[i, j] = v
        cramers_matrix.iloc[j, i] = v


# ------------------------------------------------------------
# Find highly associated pairs
# ------------------------------------------------------------
high_pairs = []

for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):

        v = cramers_matrix.iloc[i, j]

        if v >= CRAMERS_V_THRESHOLD:
            high_pairs.append({
                "Feature 1": feature_cols[i],
                "Feature 2": feature_cols[j],
                "Cramer's V": v
            })

high_pairs_df = pd.DataFrame(high_pairs)

print(
    f"=== HIGHLY ASSOCIATED FEATURE PAIRS "
    f"(V >= {CRAMERS_V_THRESHOLD}) ==="
)

if len(high_pairs_df) > 0:
    display(
        high_pairs_df.sort_values(
            "Cramer's V",
            ascending=False
        ).style.format({
            "Cramer's V": "{:.4f}"
        })
    )
else:
    print("No highly associated feature pairs found.")


# ------------------------------------------------------------
# Select redundant features to remove
# ------------------------------------------------------------
features_to_drop = set()

for pair in high_pairs:

    f1 = pair["Feature 1"]
    f2 = pair["Feature 2"]

    if f1 in features_to_drop or f2 in features_to_drop:
        continue

    mean_v_f1 = cramers_matrix.loc[
        f1,
        cramers_matrix.columns != f1
    ].mean()

    mean_v_f2 = cramers_matrix.loc[
        f2,
        cramers_matrix.columns != f2
    ].mean()

    if mean_v_f1 > mean_v_f2:
        features_to_drop.add(f1)
    else:
        features_to_drop.add(f2)

features_to_drop = sorted(features_to_drop)

print("\nFeatures to drop:")
print(features_to_drop)

# Final reduced dataset
df_npha = df_work.drop(
    columns=features_to_drop
).copy()

print("\n=== REDUCED DATASET ===")
print(f"Rows               : {len(df_npha)}")
print(f"Remaining features : {df_npha.shape[1] - 1}")
print(f"Target             : {TARGET}")

=== HIGHLY ASSOCIATED FEATURE PAIRS (V >= 0.65) ===
No highly associated feature pairs found.

Features to drop:
[]

=== REDUCED DATASET ===
Rows               : 714
Remaining features : 13
Target             : Number_of_Doctors_Visited


In [8]:
# ============================================================
# Cell 5B: Save NPHA dataset for AVISTA
# ============================================================

from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

avista_path = data_dir / "uci_npha_avista.csv"

df_npha.to_csv(
    avista_path,
    index=False
)

# Verify
df_check = pd.read_csv(avista_path)

print(f"Saved to      : {avista_path}")
print(f"Rows          : {df_check.shape[0]}")
print(f"Features      : {df_check.shape[1] - 1}")
print(f"Total columns : {df_check.shape[1]}")
print(f"Target        : {TARGET}")

Saved to      : data\uci_npha_avista.csv
Rows          : 714
Features      : 13
Total columns : 14
Target        : Number_of_Doctors_Visited


In [9]:
# ============================================================
# Cell 6: Recreate feature-pattern groups
# ============================================================

feature_cols = [
    col for col in df_npha.columns
    if col != TARGET
]

df_npha["pattern_id"] = (
    df_npha
    .groupby(feature_cols, dropna=False)
    .ngroup()
)

print("=== FEATURE-PATTERN GROUPS ===")
print(f"Total observations     : {len(df_npha)}")
print(f"Unique feature patterns: {df_npha['pattern_id'].nunique()}")

group_sizes = df_npha["pattern_id"].value_counts()

print(f"Patterns appearing once: {(group_sizes == 1).sum()}")
print(f"Repeated patterns      : {(group_sizes > 1).sum()}")
print(f"Largest pattern size   : {group_sizes.max()}")

=== FEATURE-PATTERN GROUPS ===
Total observations     : 714
Unique feature patterns: 634
Patterns appearing once: 586
Repeated patterns      : 48
Largest pattern size   : 5


In [10]:
# ============================================================
# Cell 7: Group-aware stratified 70/10/20 split
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_SEED = 42

# ------------------------------------------------------------
# Features, target, and grouping variable
# ------------------------------------------------------------
X_all = df_npha.drop(
    columns=[TARGET, "pattern_id"]
).copy()

y_all = df_npha[TARGET].copy()
groups = df_npha["pattern_id"].copy()

# ------------------------------------------------------------
# Create 10 stratified group-aware folds
#
# 7 folds -> training
# 1 fold  -> validation
# 2 folds -> test
# ------------------------------------------------------------
sgkf = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_SEED
)

fold_id = np.empty(len(df_npha), dtype=int)

for fold, (_, fold_idx) in enumerate(
    sgkf.split(X_all, y_all, groups)
):
    fold_id[fold_idx] = fold

# Assign folds
test_folds = [0, 1]
val_folds = [2]

train_mask = ~np.isin(
    fold_id,
    test_folds + val_folds
)

val_mask = np.isin(
    fold_id,
    val_folds
)

test_mask = np.isin(
    fold_id,
    test_folds
)

# ------------------------------------------------------------
# Final split
# ------------------------------------------------------------
X_train = X_all.iloc[
    np.where(train_mask)[0]
].copy()

y_train = y_all.iloc[
    np.where(train_mask)[0]
].copy()

X_val = X_all.iloc[
    np.where(val_mask)[0]
].copy()

y_val = y_all.iloc[
    np.where(val_mask)[0]
].copy()

X_test = X_all.iloc[
    np.where(test_mask)[0]
].copy()

y_test = y_all.iloc[
    np.where(test_mask)[0]
].copy()

# ------------------------------------------------------------
# Display split sizes
# ------------------------------------------------------------
print("=== GROUP-AWARE DATA SPLIT ===")
print(f"Random seed : {RANDOM_SEED}")

print(
    f"Training    : {len(X_train)} "
    f"({len(X_train)/len(df_npha)*100:.2f}%)"
)

print(
    f"Validation  : {len(X_val)} "
    f"({len(X_val)/len(df_npha)*100:.2f}%)"
)

print(
    f"Test        : {len(X_test)} "
    f"({len(X_test)/len(df_npha)*100:.2f}%)"
)

# ------------------------------------------------------------
# Class distributions
# ------------------------------------------------------------
print("\n=== CLASS DISTRIBUTION ===")

for name, target in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")

    summary = pd.DataFrame({
        "Count": target.value_counts().sort_index(),
        "Percent": (
            target.value_counts(normalize=True)
            .sort_index() * 100
        ).round(2)
    })

    display(summary)

# ------------------------------------------------------------
# Verify zero group overlap
# ------------------------------------------------------------
train_groups = set(groups.iloc[np.where(train_mask)[0]])
val_groups = set(groups.iloc[np.where(val_mask)[0]])
test_groups = set(groups.iloc[np.where(test_mask)[0]])

print("=== GROUP OVERLAP CHECK ===")
print("Train ↔ Validation :", len(train_groups & val_groups))
print("Train ↔ Test       :", len(train_groups & test_groups))
print("Validation ↔ Test  :", len(val_groups & test_groups))

=== GROUP-AWARE DATA SPLIT ===
Random seed : 42
Training    : 494 (69.19%)
Validation  : 72 (10.08%)
Test        : 148 (20.73%)

=== CLASS DISTRIBUTION ===

Training


,Count,Percent
Number_of_Doctors_Visited,,
1,91,18.42
2,249,50.40
3,154,31.17



Validation


,Count,Percent
Number_of_Doctors_Visited,,
1,14,19.44
2,37,51.39
3,21,29.17



Test


,Count,Percent
Number_of_Doctors_Visited,,
1,26,17.57
2,86,58.11
3,36,24.32


=== GROUP OVERLAP CHECK ===
Train ↔ Validation : 0
Train ↔ Test       : 0
Validation ↔ Test  : 0


In [11]:
# ============================================================
# Cell 8: AVISTA-equivalent categorical preprocessing
# Fit on TRAIN only
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_features = X_train.columns.tolist()

print(f"Categorical features : {len(categorical_features)}")
print("Numerical features   : 0")

# ------------------------------------------------------------
# Categorical preprocessing
# Missing -> Unknown
# One-hot encoding
# ------------------------------------------------------------
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

# ------------------------------------------------------------
# Fit ONLY on training data
# ------------------------------------------------------------
X_train_processed = preprocessor.fit_transform(X_train)

# Validation/test transformed using train-fitted artifact
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("\n=== PROCESSED DATA ===")
print("Train      :", X_train_processed.shape)
print("Validation :", X_val_processed.shape)
print("Test       :", X_test_processed.shape)

print("\nPreprocessing fitted on training data only.")

Categorical features : 13
Numerical features   : 0

=== PROCESSED DATA ===
Train      : (494, 48)
Validation : (72, 48)
Test       : (148, 48)

Preprocessing fitted on training data only.


In [12]:
# ============================================================
# Cell 9: Multiclass SMOTE Moderate (0.5)
# Apply to TRAIN only
# ============================================================

import numpy as np
from imblearn.over_sampling import SMOTE

SMOTE_RATIO = 0.5
SMOTE_SEED = 42

# ------------------------------------------------------------
# Class counts before SMOTE
# ------------------------------------------------------------
class_counts = y_train.value_counts().sort_index()

majority_count = class_counts.max()
target_count = int(np.floor(SMOTE_RATIO * majority_count))

print("=== BEFORE SMOTE ===")
print(class_counts)

print(f"\nMajority class count : {majority_count}")
print(f"Moderate 0.5 target  : {target_count}")

# ------------------------------------------------------------
# Build sampling strategy
# Only classes below the target are oversampled
# ------------------------------------------------------------
sampling_strategy = {
    cls: target_count
    for cls, count in class_counts.items()
    if count < target_count
}

print("\nSampling strategy:")
print(sampling_strategy)

# ------------------------------------------------------------
# Apply SMOTE only to training data
# ------------------------------------------------------------
if sampling_strategy:

    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=SMOTE_SEED
    )

    X_train_final, y_train_final = smote.fit_resample(
        X_train_processed,
        y_train
    )

    print("\nSMOTE applied.")

else:
    X_train_final = X_train_processed
    y_train_final = y_train.copy()

    print("\nSMOTE not required.")

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
print("\n=== AFTER SMOTE ===")
print(y_train_final.value_counts().sort_index())

print(f"\nOriginal training N  : {len(y_train)}")
print(f"Effective training N : {len(y_train_final)}")

=== BEFORE SMOTE ===
Number_of_Doctors_Visited
1     91
2    249
3    154
Name: count, dtype: int64

Majority class count : 249
Moderate 0.5 target  : 124

Sampling strategy:
{1: 124}

SMOTE applied.

=== AFTER SMOTE ===
Number_of_Doctors_Visited
1    124
2    249
3    154
Name: count, dtype: int64

Original training N  : 494
Effective training N : 527


%USERPROFILE%\anaconda3\envs\mimi\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "%USERPROFILE%\anaconda3\envs\mimi\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "%USERPROFILE%\anaconda3\envs\mimi\Lib\subprocess.py", line 550, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "%USERPROFILE%\anaconda3\envs\mimi\Lib\subprocess.py", line 1028, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "%USERPROFILE%\anaconda3\envs\mimi\Lib\subprocess.py", line 1540, in _execute_c

In [13]:
# ============================================================
# Cell 10: Group-aware 5-Fold CV
# Logistic Regression + Random Forest + XGBoost
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE


RANDOM_SEED = 42
N_FOLDS = 5
SMOTE_RATIO = 0.5


# ------------------------------------------------------------
# Training-set groups from the original group-aware split
# ------------------------------------------------------------
groups_train = groups.iloc[
    np.where(train_mask)[0]
].copy()


# ------------------------------------------------------------
# Encode multiclass target for XGBoost
# Classes: 1, 2, 3 -> 0, 1, 2
# ------------------------------------------------------------
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

print("Target classes:", label_encoder.classes_)


# ------------------------------------------------------------
# Group-aware CV
# ------------------------------------------------------------
cv = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)


models_results = {
    "Logistic Regression": [],
    "Random Forest": [],
    "XGBoost": []
}


for fold, (train_idx, fold_val_idx) in enumerate(
    cv.split(X_train, y_train, groups_train),
    start=1
):

    print(f"\nRunning fold {fold}...")

    # ========================================================
    # Fold data
    # ========================================================

    X_fold_train = X_train.iloc[train_idx].copy()
    y_fold_train = y_train.iloc[train_idx].copy()

    X_fold_val = X_train.iloc[fold_val_idx].copy()
    y_fold_val = y_train.iloc[fold_val_idx].copy()

    # ========================================================
    # Preprocessing — fit on fold TRAIN only
    # ========================================================

    fold_preprocessor = ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="Unknown"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )
                )
            ]),
            X_fold_train.columns.tolist()
        )
    ])

    X_fold_train_processed = (
        fold_preprocessor.fit_transform(X_fold_train)
    )

    X_fold_val_processed = (
        fold_preprocessor.transform(X_fold_val)
    )

    # ========================================================
    # SMOTE Moderate 0.5 — fold TRAIN only
    # ========================================================

    counts = y_fold_train.value_counts()

    majority_count = counts.max()

    target_count = int(
        np.floor(SMOTE_RATIO * majority_count)
    )

    sampling_strategy = {
        cls: target_count
        for cls, count in counts.items()
        if count < target_count
    }

    if sampling_strategy:

        smote = SMOTE(
            sampling_strategy=sampling_strategy,
            random_state=RANDOM_SEED
        )

        X_fit, y_fit = smote.fit_resample(
            X_fold_train_processed,
            y_fold_train
        )

    else:

        X_fit = X_fold_train_processed
        y_fit = y_fold_train.copy()

    # ========================================================
    # 1. Logistic Regression
    # ========================================================

    lr = LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=1000
    )

    start = time.perf_counter()

    lr.fit(X_fit, y_fit)

    runtime = time.perf_counter() - start

    pred = lr.predict(X_fold_val_processed)
    prob = lr.predict_proba(X_fold_val_processed)

    models_results["Logistic Regression"].append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, pred),
        "Macro-F1": f1_score(
            y_fold_val, pred, average="macro"
        ),
        "Precision": precision_score(
            y_fold_val,
            pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_fold_val,
            pred,
            average="macro",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_fold_val,
            prob,
            multi_class="ovr",
            average="macro"
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "Runtime (sec)": runtime
    })

    # ========================================================
    # 2. Random Forest
    # ========================================================

    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    start = time.perf_counter()

    rf.fit(X_fit, y_fit)

    runtime = time.perf_counter() - start

    pred = rf.predict(X_fold_val_processed)
    prob = rf.predict_proba(X_fold_val_processed)

    models_results["Random Forest"].append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, pred),
        "Macro-F1": f1_score(
            y_fold_val, pred, average="macro"
        ),
        "Precision": precision_score(
            y_fold_val,
            pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_fold_val,
            pred,
            average="macro",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_fold_val,
            prob,
            multi_class="ovr",
            average="macro"
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "Runtime (sec)": runtime
    })

    # ========================================================
    # 3. XGBoost
    # ========================================================

    y_fit_xgb = label_encoder.transform(y_fit)
    y_val_xgb = label_encoder.transform(y_fold_val)

    xgb = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        random_state=RANDOM_SEED,
        eval_metric="mlogloss",
        n_jobs=-1
    )

    start = time.perf_counter()

    xgb.fit(X_fit, y_fit_xgb)

    runtime = time.perf_counter() - start

    pred = xgb.predict(X_fold_val_processed)
    prob = xgb.predict_proba(X_fold_val_processed)

    models_results["XGBoost"].append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_val_xgb, pred),
        "Macro-F1": f1_score(
            y_val_xgb, pred, average="macro"
        ),
        "Precision": precision_score(
            y_val_xgb,
            pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_val_xgb,
            pred,
            average="macro",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_val_xgb,
            prob,
            multi_class="ovr",
            average="macro"
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "Runtime (sec)": runtime
    })


# ============================================================
# CV Summary
# ============================================================

metrics = [
    "Accuracy",
    "Macro-F1",
    "Precision",
    "Recall",
    "ROC-AUC"
]

for model_name, results in models_results.items():

    result_df = pd.DataFrame(results)

    print(f"\n=== {model_name.upper()} — 5-FOLD CV ===")

    display(result_df)

    print("Summary:")

    for metric in metrics:
        print(
            f"{metric:10s}: "
            f"{result_df[metric].mean():.6f} "
            f"± {result_df[metric].std():.6f}"
        )

Target classes: [1 2 3]

Running fold 1...

Running fold 2...

Running fold 3...

Running fold 4...

Running fold 5...

=== LOGISTIC REGRESSION — 5-FOLD CV ===


,Fold,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Train N,Effective N,Runtime (sec)
0,1,0.484848,0.416517,0.453598,0.416111,0.552770,395,427,0.040360
1,2,0.466667,0.375123,0.388645,0.378816,0.621287,389,406,0.021447
2,3,0.427184,0.330919,0.346032,0.332080,0.539825,391,415,0.021431
3,4,0.412371,0.287914,0.289525,0.298047,0.502667,397,423,0.018324
4,5,0.500000,0.391667,0.569820,0.435185,0.597479,404,437,0.021454


Summary:
Accuracy  : 0.458214 ± 0.037388
Macro-F1  : 0.360428 ± 0.051154
Precision : 0.409524 ± 0.107824
Recall    : 0.372048 ± 0.057112
ROC-AUC   : 0.562806 ± 0.047084

=== RANDOM FOREST — 5-FOLD CV ===


,Fold,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Train N,Effective N,Runtime (sec)
0,1,0.494949,0.426986,0.480328,0.435556,0.646522,395,427,0.207188
1,2,0.533333,0.428270,0.452650,0.428507,0.563776,389,406,0.201988
2,3,0.436893,0.317509,0.405090,0.320802,0.484468,391,415,0.181551
3,4,0.432990,0.282078,0.266667,0.306415,0.529773,397,423,0.185043
4,5,0.377778,0.255304,0.250415,0.314815,0.483025,404,437,0.171890


Summary:
Accuracy  : 0.455189 ± 0.060223
Macro-F1  : 0.342029 ± 0.081197
Precision : 0.371030 ± 0.106310
Recall    : 0.361219 ± 0.064892
ROC-AUC   : 0.541513 ± 0.067704

=== XGBOOST — 5-FOLD CV ===


,Fold,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Train N,Effective N,Runtime (sec)
0,1,0.454545,0.360411,0.385510,0.407222,0.601901,395,427,0.502345
1,2,0.390476,0.340672,0.357727,0.353563,0.540586,389,406,0.225347
2,3,0.436893,0.336561,0.345771,0.338346,0.483731,391,415,0.228830
3,4,0.463918,0.347100,0.340038,0.357670,0.552972,397,423,0.189635
4,5,0.388889,0.313512,0.317962,0.333333,0.516289,404,437,0.170875


Summary:
Accuracy  : 0.426944 ± 0.035376
Macro-F1  : 0.339651 ± 0.017178
Precision : 0.349402 ± 0.024811
Recall    : 0.358027 ± 0.029314
ROC-AUC   : 0.539096 ± 0.043945


In [14]:
# ============================================================
# Cell 11: Final evaluation on untouched 20% test set
# Logistic Regression + Random Forest + XGBoost
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE


RANDOM_SEED = 42
SMOTE_RATIO = 0.5


# ------------------------------------------------------------
# 1. Fit preprocessing on FULL 70% training set only
# ------------------------------------------------------------

final_preprocessor = ColumnTransformer([
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Unknown"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]),
        X_train.columns.tolist()
    )
])

X_train_processed = final_preprocessor.fit_transform(X_train)
X_test_processed = final_preprocessor.transform(X_test)


# ------------------------------------------------------------
# 2. SMOTE Moderate (0.5) on TRAIN only
# ------------------------------------------------------------

counts = y_train.value_counts()
majority_count = counts.max()

target_count = int(
    np.floor(SMOTE_RATIO * majority_count)
)

sampling_strategy = {
    cls: target_count
    for cls, count in counts.items()
    if count < target_count
}

if sampling_strategy:

    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=RANDOM_SEED
    )

    X_train_final, y_train_final = smote.fit_resample(
        X_train_processed,
        y_train
    )

else:

    X_train_final = X_train_processed
    y_train_final = y_train.copy()


print("=== FINAL TRAINING DATA ===")
print(f"Original training N  : {len(y_train)}")
print(f"Effective training N : {len(y_train_final)}")
print(f"Test N               : {len(y_test)}")

print("\nClass counts after SMOTE:")
print(y_train_final.value_counts().sort_index())


# ------------------------------------------------------------
# 3. Label encoder for XGBoost
# ------------------------------------------------------------

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train_xgb = label_encoder.transform(y_train_final)
y_test_xgb = label_encoder.transform(y_test)


# ------------------------------------------------------------
# 4. Models
# ------------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=1000
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        random_state=RANDOM_SEED,
        eval_metric="mlogloss",
        n_jobs=-1
    )
}


# ------------------------------------------------------------
# 5. Final test evaluation
# ------------------------------------------------------------

results = []

for model_name, model in models.items():

    start = time.perf_counter()

    if model_name == "XGBoost":

        model.fit(
            X_train_final,
            y_train_xgb
        )

        runtime = time.perf_counter() - start

        y_pred = model.predict(X_test_processed)
        y_prob = model.predict_proba(X_test_processed)

        accuracy = accuracy_score(y_test_xgb, y_pred)

        macro_f1 = f1_score(
            y_test_xgb,
            y_pred,
            average="macro"
        )

        precision = precision_score(
            y_test_xgb,
            y_pred,
            average="macro",
            zero_division=0
        )

        recall = recall_score(
            y_test_xgb,
            y_pred,
            average="macro",
            zero_division=0
        )

        roc_auc = roc_auc_score(
            y_test_xgb,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    else:

        model.fit(
            X_train_final,
            y_train_final
        )

        runtime = time.perf_counter() - start

        y_pred = model.predict(X_test_processed)
        y_prob = model.predict_proba(X_test_processed)

        accuracy = accuracy_score(y_test, y_pred)

        macro_f1 = f1_score(
            y_test,
            y_pred,
            average="macro"
        )

        precision = precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        roc_auc = roc_auc_score(
            y_test,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro-F1": macro_f1,
        "Precision": precision,
        "Recall": recall,
        "ROC-AUC": roc_auc,
        "Training N": len(y_train),
        "Effective N": len(y_train_final),
        "Test N": len(y_test),
        "Runtime (sec)": runtime
    })


# ------------------------------------------------------------
# 6. Results table
# ------------------------------------------------------------

test_results = pd.DataFrame(results)

display(
    test_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Runtime (sec)": "{:.4f}"
    })
)

=== FINAL TRAINING DATA ===
Original training N  : 494
Effective training N : 527
Test N               : 148

Class counts after SMOTE:
Number_of_Doctors_Visited
1    124
2    249
3    154
Name: count, dtype: int64


,Model,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Effective N,Test N,Runtime (sec)
0,Logistic Regression,0.452703,0.355792,0.365632,0.352862,0.558225,494,527,148,0.0288
1,Random Forest,0.466216,0.290430,0.277160,0.310508,0.534805,494,527,148,0.2637
2,XGBoost,0.459459,0.382465,0.418561,0.378271,0.537811,494,527,148,0.2742


In [15]:
# ============================================================
# Cell 12: AutoGluon — NPHA
# LR + Random Forest + XGBoost
# Same 70/10/20 split + group-aware 5-fold bagging
# ============================================================

import time
import pandas as pd

from autogluon.tabular import TabularPredictor

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

TARGET = "Number_of_Doctors_Visited"


# ============================================================
# 1. Recover pattern groups for each existing split
# ============================================================

groups_train = groups.iloc[
    np.where(train_mask)[0]
].copy()

groups_val = groups.iloc[
    np.where(val_mask)[0]
].copy()

groups_test = groups.iloc[
    np.where(test_mask)[0]
].copy()


# ============================================================
# 2. Build RAW AutoGluon datasets
#
# No one-hot encoding
# No manual imputation
# No SMOTE
# pattern_id is used ONLY for group-aware CV
# ============================================================

ag_train = X_train.copy()
ag_train[TARGET] = y_train.values
ag_train["pattern_id"] = groups_train.values

ag_val = X_val.copy()
ag_val[TARGET] = y_val.values
ag_val["pattern_id"] = groups_val.values

ag_test = X_test.copy()
ag_test[TARGET] = y_test.values
ag_test["pattern_id"] = groups_test.values


print("=== AUTOGLUON DATA ===")
print(f"Training    : {len(ag_train)}")
print(f"Validation  : {len(ag_val)}")
print(f"Test        : {len(ag_test)}")

print("\nTraining class distribution:")
print(ag_train[TARGET].value_counts().sort_index())


# ============================================================
# 3. Train AutoGluon
#
# LR  = Logistic Regression / LinearModel
# RF  = Random Forest
# XGB = XGBoost
#
# pattern_id is ignored as a predictor.
# ============================================================

start_time = time.perf_counter()

predictor_npha = TabularPredictor(
    label=TARGET,
    problem_type="multiclass",
    eval_metric="accuracy",

    # Prevent pattern_id from becoming a predictor
    learner_kwargs={
        "ignored_columns": ["pattern_id"]
    },

    path="AutogluonModels/npha_LR_RF_XGB_5fold"
).fit(

    train_data=ag_train,

    # Same external 10% validation partition
    tuning_data=ag_val,

    # Only the same three model families
    hyperparameters={
        "LR": {},
        "RF": {},
        "XGB": {}
    },

    # --------------------------------------------------------
    # Group-aware 5-fold bagging
    # --------------------------------------------------------
    num_bag_folds=5,
    num_bag_sets=1,

    validation_structure={
        "group_on": "pattern_id"
    },

    # Use supplied 10% validation set as external holdout
    use_bag_holdout=True,

    # No stacking / weighted ensemble
    num_stack_levels=0,
    fit_weighted_ensemble=False,
    fit_full_last_level_weighted_ensemble=False
)

total_training_runtime = (
    time.perf_counter() - start_time
)


# ============================================================
# 4. AutoGluon leaderboard
# ============================================================

print("\n=== AUTOGLUON LEADERBOARD ===")

leaderboard = predictor_npha.leaderboard(
    ag_test,
    extra_info=False
)

display(leaderboard)


# ============================================================
# 5. Final evaluation of each model on untouched 20% test set
# ============================================================

# Keep pattern_id present; AutoGluon knows it is an ignored column
X_ag_test = ag_test.drop(columns=TARGET)
y_ag_test = ag_test[TARGET]

class_labels = list(predictor_npha.class_labels)

print("\nClass order used by AutoGluon:")
print(class_labels)


results = []

for model_name in leaderboard["model"].tolist():

    start_pred = time.perf_counter()

    y_pred = predictor_npha.predict(
        X_ag_test,
        model=model_name
    )

    y_prob = predictor_npha.predict_proba(
        X_ag_test,
        model=model_name
    )

    prediction_runtime = (
        time.perf_counter() - start_pred
    )

    # Ensure probability columns follow AutoGluon's class order
    if isinstance(y_prob, pd.DataFrame):
        y_prob_ordered = y_prob[class_labels].values
    else:
        y_prob_ordered = y_prob

    results.append({

        "Model": model_name,

        "Accuracy": accuracy_score(
            y_ag_test,
            y_pred
        ),

        "Macro-F1": f1_score(
            y_ag_test,
            y_pred,
            average="macro"
        ),

        "Precision": precision_score(
            y_ag_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall": recall_score(
            y_ag_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_ag_test,
            y_prob_ordered,
            labels=class_labels,
            multi_class="ovr",
            average="macro"
        ),

        "Training N": len(ag_train),

        "Validation N": len(ag_val),

        "Test N": len(ag_test),

        "CV Folds": 5,

        "SMOTE": "No",

        "Prediction Runtime (sec)": prediction_runtime
    })


# ============================================================
# 6. Final results
# ============================================================

ag_npha_results = pd.DataFrame(results)

print(
    "\n=== AUTOGLUON — NPHA FINAL TEST RESULTS ==="
)

display(
    ag_npha_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Prediction Runtime (sec)": "{:.4f}"
    })
)

print(
    f"\nTotal AutoGluon training runtime: "
    f"{total_training_runtime:.3f} sec"
)

Verbosity: 2 (Standard Logging)


=== AUTOGLUON DATA ===
Training    : 494
Validation  : 72
Test        : 148

Training class distribution:
Number_of_Doctors_Visited
1     91
2    249
3    154
Name: count, dtype: int64


=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.6.0+cu118
CUDA Version:       11.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       20.23 GB / 47.93 GB (42.2%)
Disk Space Avail:   267.35 GB / 610.93 GB (43.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabular Foundation Models (


=== AUTOGLUON LEADERBOARD ===


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest_BAG_L1,0.493243,0.416667,accuracy,0.752523,0.639750,1.195931,0.752523,0.639750,1.195931,1,True,1
1,XGBoost_BAG_L1,0.479730,0.458333,accuracy,0.210001,0.373522,2.178854,0.210001,0.373522,2.178854,1,True,2
2,LinearModel_BAG_L1,0.472973,0.486111,accuracy,0.051075,0.147799,4.018807,0.051075,0.147799,4.018807,1,True,3



Class order used by AutoGluon:
[1, 2, 3]

=== AUTOGLUON — NPHA FINAL TEST RESULTS ===


,Model,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Validation N,Test N,CV Folds,SMOTE,Prediction Runtime (sec)
0,RandomForest_BAG_L1,0.493243,0.328892,0.343940,0.345723,0.564897,494,72,148,5,No,0.9101
1,XGBoost_BAG_L1,0.479730,0.325963,0.348908,0.343355,0.581998,494,72,148,5,No,0.4246
2,LinearModel_BAG_L1,0.472973,0.311741,0.351852,0.328712,0.556943,494,72,148,5,No,0.0908



Total AutoGluon training runtime: 30.977 sec
